# 06 — Split the data across hospitals

The federated experiments need two things that do not exist yet: one global test
set that every run is scored on, and one folder per hospital per scenario holding
only that hospital's patients.

This notebook builds both, and it is the only place the word "hospital" means a
directory. Nothing here trains anything.

**The rule everything else depends on.** A patient goes to exactly one hospital,
with all of their slices. Splitting by slice instead would let a model recognise the
patient rather than the disease, and every number after that would be inflated.

**No image is ever regenerated.** The PNGs built in notebook 02 are hardlinked into
the hospital folders, so the bytes a hospital trains on are the same inodes as the
bytes in `dataset/multi_subtype_80mm/`. Nothing is resampled, re-cropped,
re-normalised or re-encoded. That is what lets the federated arm and the centralised
baseline be compared at all.

Hardlinks cannot cross filesystems. If the dataset lives on a RAM disk and the
output does not, this falls back to copying, and it will say so.

## Configuration

Every path and every number for the whole federated campaign. The training
hyperparameters are further down and they are deliberately identical to notebook
03: if the federated clients trained with different regularisation, the gap being
measured would be the regularisation rather than the federation.

In [ ]:
from pathlib import Path

# Repository root. The notebook lives in notebooks/, so the parent is the root.
REPO_ROOT = Path.cwd().parent

# INPUT, must exist. Built by notebook 02. Expect train.csv, val.csv, test.csv and
# an images/ tree with one folder per patient.
SOURCE_DATASET = REPO_ROOT / "dataset" / "multi_subtype_80mm"
SOURCE_IMAGES = SOURCE_DATASET / "images"

# OUTPUT root for everything the deployment reads.
DATA_DIR = REPO_ROOT / "deployment" / "data"

# OUTPUT. The held-out global test set. Identical for all thirteen experiments,
# and the only set any reported number comes from. No hospital ever sees it.
GLOBAL_DIR = DATA_DIR / "global"

# OUTPUT. One subfolder per scenario, each holding one folder per hospital, each
# holding that hospital's train.csv, val.csv, images/ and manifest.json.
PARTITIONS_DIR = DATA_DIR / "partitions"

# --------------------------------------------------------------------------- #
# THE TASK
# --------------------------------------------------------------------------- #

CLASS_NAMES = ["HRposHER2neg", "TripleNeg", "HER2pos"]
NUM_CLASSES = len(CLASS_NAMES)

# --------------------------------------------------------------------------- #
# SPLITTING
# --------------------------------------------------------------------------- #

SEED = 42                    # the same seed the training uses
LOCAL_VAL_FRACTION = 0.2     # fraction of each hospital's patients held out locally.
                             # Serves the per-round convergence curve and the metric
                             # the server selects on. The OFFICIAL number always
                             # comes from the global test set instead.

HARDLINK = True              # hardlink instead of copying. Each site still has its
                             # own path and still cannot read another's folder; this
                             # only avoids storing the same immutable PNG 13 times.
# HARDLINK = False           # real copies. Needed when source and target are on
                             # different filesystems, e.g. a tmpfs RAM disk.

FORCE_REBUILD = False        # True wipes and rebuilds an existing layout

print(f"source      {SOURCE_DATASET}   exists: {SOURCE_DATASET.is_dir()}")
print(f"global test {GLOBAL_DIR}")
print(f"partitions  {PARTITIONS_DIR}")

## The six partitions

Each row is one physical way of dividing the training patients. `ratio` is the
shape of the split, not a set of fractions: the thesis describes the skewed case as
50/20/10/10, which sums to 90 rather than 100, so it is stored as the 5:2:1:1 ratio
it actually is and normalised here. That keeps the wording and the behaviour in
agreement instead of silently disagreeing by ten percent.

The first four are stratified, meaning each hospital's class ratio matches the
global one. Only quantity varies. That is quantity skew, the weakest entry in the
non-IID taxonomy, and stating it as a limitation is more honest than calling it
heterogeneity.

The last two are the pair that answers RQ2, and they only mean anything against
each other. Both hold 642, 101 and 784 patients. The only difference is whether a
hospital draws from one cohort or from all three, so anything measured between them
is attributable to cohort identity rather than to site size. The class-share spread
is 27.5 percentage points against 0.3.

The order matters in the cohort split: sorted alphabetically, DUKE goes to
hospital_1, I-SPY1 to hospital_2 and I-SPY2 to hospital_3. The size-matched control
repeats those sizes in that order, or the two stop being size-matched.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Partition:
    name: str
    n_clients: int
    ratio: tuple           # the SHAPE of the split, normalised below
    label: str
    stratified: bool = True     # False = one real cohort per hospital

    @property
    def fractions(self):
        total = sum(self.ratio)
        return tuple(r / total for r in self.ratio)

    @property
    def client_names(self):
        return [f"hospital_{i + 1}" for i in range(self.n_clients)]

    def describe(self):
        pct = " / ".join(f"{100 * f:.1f}%" for f in self.fractions)
        return f"{self.label}  ->  {pct}"


PARTITIONS = {
    "2_clients_balanced": Partition(
        "2_clients_balanced", 2, (1, 1), "2 hospitals, balanced (50/50)"),
    "3_clients_balanced": Partition(
        "3_clients_balanced", 3, (1, 1, 1), "3 hospitals, balanced (33.3 each)"),
    "4_clients_balanced": Partition(
        "4_clients_balanced", 4, (1, 1, 1, 1), "4 hospitals, balanced (25 each)"),
    "4_clients_skewed": Partition(
        "4_clients_skewed", 4, (5, 2, 1, 1),
        "4 hospitals, skewed 5:2:1:1 (the thesis writes it 50/20/10/10)"),

    # The RQ2 pair. Compared against each other and against nothing else.
    "3_clients_cohort": Partition(
        "3_clients_cohort", 3, (642, 101, 784),
        "3 hospitals, one cohort each (DUKE | I-SPY1 | I-SPY2)", stratified=False),
    "3_clients_sizematched": Partition(
        "3_clients_sizematched", 3, (642, 101, 784),
        "3 hospitals, cohorts mixed, sizes matched to 3_clients_cohort"),
}

# Which ones to build. All six is what the campaign used.
BUILD = list(PARTITIONS)
# BUILD = ["3_clients_cohort", "3_clients_sizematched"]   # just the RQ2 pair

for name in BUILD:
    print(f"  {name:<24} {PARTITIONS[name].describe()}")

## Imports

In [ ]:
import json
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

assert (SOURCE_DATASET / "train.csv").is_file(), (
    f"{SOURCE_DATASET} is not a prepared dataset (no train.csv). Run notebook 02.")


def portable(path):
    """A path relative to the repository root, for recording in a manifest.

    Manifests are provenance records that travel with the repository, so an
    absolute path in one is wrong the moment anybody clones it elsewhere, or the
    moment the folder is renamed.
    """
    path = Path(path).resolve()
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


print("ready")

## Placing images

One patient at a time, and it raises if a patient has no image folder rather than
quietly producing a hospital with fewer images than its CSV claims.

The hardlink is the whole reason this is cheap. Thirteen experiments across six
partitions would otherwise mean storing the same PNG many times over. A hardlink is
a second name for the same inode, so the disk cost is one copy and each hospital
still has its own path and still cannot read another hospital's folder.

In [ ]:
def copy_patient_images(pids, src_images, dst_images, hardlink):
    """Place one patient's slices at a time. Returns the number of files placed."""
    n = 0
    for pid in pids:
        src = src_images / str(pid)
        dst = dst_images / str(pid)
        if not src.is_dir():
            raise FileNotFoundError(f"no image folder for patient {pid}: {src}")
        dst.mkdir(parents=True, exist_ok=True)
        for png in src.glob("*.png"):
            target = dst / png.name
            if target.exists():
                continue
            if hardlink:
                # Hardlinks cannot cross filesystems. A dataset on a RAM disk and
                # an output on the SSD will raise here; fall back to copying.
                try:
                    target.hardlink_to(png)
                except OSError:
                    shutil.copy2(png, target)
            else:
                shutil.copy2(png, target)
            n += 1
    return n


print("image placement defined")

## The global test set

268 patients, held out, identical for all thirteen experiments. Every reported
number in the thesis comes from here and from nowhere else.

The validation split is placed alongside it, because the collection step in
notebook 07 scores against the same layout.

No hospital ever receives this folder. It exists on the coordinating machine only.

In [ ]:
def write_split(rows, out_dir, split, src_images, hardlink):
    rows = rows.copy()
    rows["split"] = split
    rows.to_csv(out_dir / f"{split}.csv", index=False)
    pids = rows.pid.unique()
    n_files = copy_patient_images(pids, src_images, out_dir / "images", hardlink)
    counts = rows.drop_duplicates("pid").label.value_counts().sort_index()
    summary = {
        "split": split, "patients": int(len(pids)), "slices": int(len(rows)),
        "files_placed": n_files,
        "per_class_patients": [int(counts.get(c, 0)) for c in range(NUM_CLASSES)],
        "trivial_baseline": float(
            rows.drop_duplicates("pid").label.value_counts().max() / len(pids)),
    }
    print(f"  {split:<5} {summary['patients']:>4} patients  "
          f"{summary['slices']:>6,} slices  per-class "
          f"{summary['per_class_patients']}  trivial "
          f"{summary['trivial_baseline']:.4f}")
    return summary


if GLOBAL_DIR.exists() and any(GLOBAL_DIR.iterdir()) and not FORCE_REBUILD:
    print(f"{GLOBAL_DIR} already built — set FORCE_REBUILD = True to redo it")
    global_manifest = json.loads((GLOBAL_DIR / "manifest.json").read_text())
    for s, m in global_manifest["splits"].items():
        print(f"  {s:<5} {m['patients']:>4} patients  {m['slices']:>6,} slices")
else:
    if GLOBAL_DIR.exists():
        shutil.rmtree(GLOBAL_DIR)
    (GLOBAL_DIR / "images").mkdir(parents=True, exist_ok=True)
    print(f"source: {SOURCE_DATASET}\ntarget: {GLOBAL_DIR}\n")

    global_manifest = {
        "source": portable(SOURCE_DATASET),
        "built": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "hardlinked": bool(HARDLINK), "classes": CLASS_NAMES, "splits": {},
    }
    for split in ("test", "val"):
        csv_path = SOURCE_DATASET / f"{split}.csv"
        if not csv_path.is_file():
            print(f"  {split:<5} not present in source — skipped")
            continue
        global_manifest["splits"][split] = write_split(
            pd.read_csv(csv_path), GLOBAL_DIR, split, SOURCE_IMAGES, HARDLINK)

    if "test" not in global_manifest["splits"]:
        raise RuntimeError(f"{SOURCE_DATASET} has no test.csv — nothing to score on")

    (GLOBAL_DIR / "manifest.json").write_text(json.dumps(global_manifest, indent=2))
    print(f"\nwritten: {GLOBAL_DIR}")

## The three ways to divide patients

**Stratified** keeps each hospital's class ratio equal to the global one, so the
only thing that varies is quantity. It is done per class rather than globally:
shuffling all patients and cutting at the fraction boundaries gives each site the
right size but lets its class ratio wander, which would silently turn a balanced
test into a mild label-skew test.

The allocation inside each class uses largest remainder. Plain rounding loses or
invents patients when the shares do not divide evenly, and 5:2:1:1 does not divide
evenly.

**Unstratified** splits by size alone and produces label skew as a side effect.

**By cohort** puts one real cohort in each hospital. This is the genuine non-IID
case: the sites differ in class prior, tumour size and scanner, not merely in how
many patients they hold.

In [ ]:
def largest_remainder(n_items, fractions):
    """How many items each share gets, without losing or inventing any."""
    exact = np.array(fractions) * n_items
    take = np.floor(exact).astype(int)
    for i in np.argsort(-(exact - take))[:n_items - take.sum()]:
        take[i] += 1
    return take


def stratified_shares(patients, fractions, rng):
    """Split patient ids across sites, keeping each site's class ratio global."""
    buckets = [[] for _ in fractions]
    for label, group in patients.groupby("label"):
        ids = group.pid.to_numpy().copy()
        rng.shuffle(ids)
        take = largest_remainder(len(ids), fractions)
        start = 0
        for i, n in enumerate(take):
            buckets[i].extend(ids[start:start + n].tolist())
            start += n
    return buckets


def unstratified_shares(patients, fractions, rng):
    """Split by size only, ignoring class. Produces label skew as a side effect."""
    ids = patients.pid.to_numpy().copy()
    rng.shuffle(ids)
    take = largest_remainder(len(ids), fractions)
    out, start = [], 0
    for n in take:
        out.append(ids[start:start + n].tolist())
        start += n
    return out


def cohort_shares(patients, n_clients):
    """One real cohort per hospital. sorted() fixes the order: duke, spy1, spy2."""
    if "cohort" not in patients.columns:
        raise RuntimeError("by-cohort needs a `cohort` column; use a pooled dataset")
    cohorts = sorted(patients.cohort.unique())
    if len(cohorts) != n_clients:
        raise RuntimeError(f"by-cohort needs one hospital per cohort: "
                           f"{len(cohorts)} cohorts but {n_clients} hospitals")
    return [patients[patients.cohort == c].pid.tolist() for c in cohorts]


print("three splitting strategies defined")

## The local validation split

Each hospital holds out a fifth of its own patients. That local split serves the
per-round convergence curve and, more importantly, the metric the server selects
the global model on.

It is stratified for a specific reason. A site holding 39 patients can easily draw a
validation split missing a whole class, and the metric code then returns NaN for
the AUC, so that site would report nothing for the very quantity the server is
selecting on.

In [ ]:
def carve_local_val(pids, patients, fraction, rng):
    """Hold out `fraction` of a site's patients, stratified, at least one per class."""
    sub = patients[patients.pid.isin(pids)]
    val = []
    for _, group in sub.groupby("label"):
        ids = group.pid.to_numpy().copy()
        rng.shuffle(ids)
        n = max(1, int(round(fraction * len(ids)))) if len(ids) > 1 else 0
        val.extend(ids[:n].tolist())
    train = [p for p in pids if p not in set(val)]
    return train, val


print("local validation carving defined")

## Build the partitions

One folder per hospital, holding its own `train.csv`, `val.csv`, `images/` and a
`manifest.json`.

Two integrity checks run at the end of every partition, and they raise rather than
warn. A patient in two hospitals is a leak. A patient count that does not match the
source means the federation would train on less data than the baseline it is
compared against, and a 3% shortfall is invisible in the output but not in the
result.

The global class weights are computed once from the pooled training split and
written into every manifest. The clients use them only when `class_weight_scope` is
"global", which the campaign leaves at "local". The distinction is RQ4 material:
local weights mean the sites optimise different objectives and the server averages
models trained on different losses, global weights mean one objective and one leaked
vector of class counts.

In [ ]:
def write_site(site_dir, rows, train_pids, val_pids, src_images, hardlink, extra):
    site_dir.mkdir(parents=True, exist_ok=True)
    summary = {"site": site_dir.name, **extra}
    for split, pids in (("train", train_pids), ("val", val_pids)):
        sub = rows[rows.pid.isin(pids)].copy()
        sub["split"] = split
        sub.to_csv(site_dir / f"{split}.csv", index=False)
        copy_patient_images(pids, src_images, site_dir / "images", hardlink)
        counts = sub.drop_duplicates("pid").label.value_counts()
        summary[split] = {
            "patients": len(pids), "slices": int(len(sub)),
            "per_class_patients": [int(counts.get(c, 0)) for c in range(NUM_CLASSES)],
        }
    (site_dir / "manifest.json").write_text(json.dumps(summary, indent=2))
    return summary


def build_partition(name, partition, rows, patients, src_images, out_root):
    rng = np.random.default_rng(SEED)
    print(f"\n{partition.describe()}")

    if not partition.stratified:
        shares = cohort_shares(patients, partition.n_clients)
        mode = "cohort"
    else:
        shares = stratified_shares(patients, partition.fractions, rng)
        mode = "stratified"
    # shares = unstratified_shares(patients, partition.fractions, rng)   # size only
    # mode = "unstratified"

    # Global class weights from the POOLED training split, so no site has to see
    # another site's data to use them.
    per_patient = patients.label.to_numpy()
    counts = np.bincount(per_patient, minlength=NUM_CLASSES).astype(float)
    counts[counts == 0] = 1.0
    global_weights = (len(per_patient) / (NUM_CLASSES * counts)).tolist()

    out_dir = out_root / name
    if out_dir.exists():
        shutil.rmtree(out_dir)

    sites, all_pids = [], []
    for i, pids in enumerate(shares):
        train_pids, val_pids = carve_local_val(pids, patients, LOCAL_VAL_FRACTION, rng)
        site = write_site(out_dir / partition.client_names[i], rows,
                          train_pids, val_pids, src_images, HARDLINK,
                          {"global_class_weights": [round(w, 6) for w in global_weights],
                           "partition": name, "mode": mode})
        sites.append(site)
        all_pids.extend(pids)
        print(f"  {site['site']:<12} train {site['train']['patients']:>4} pat "
              f"{str(site['train']['per_class_patients']):<16} "
              f"val {site['val']['patients']:>3} pat "
              f"{site['train']['slices'] + site['val']['slices']:>6,} slices")

    # These raise. A silent leak or a lost patient is worse than a failed build.
    if len(all_pids) != len(set(all_pids)):
        raise RuntimeError(f"{name}: a patient landed in two hospitals")
    if len(all_pids) != len(patients):
        raise RuntimeError(f"{name}: {len(all_pids)} patients placed of "
                           f"{len(patients)} available")

    meta = {
        "partition": name, "mode": mode, "n_clients": partition.n_clients,
        "ratio": list(partition.ratio), "fractions": list(partition.fractions),
        "seed": SEED, "source": portable(src_images.parent),
        "built": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "local_val_fraction": LOCAL_VAL_FRACTION,
        "global_class_weights": global_weights,
        "total_patients": len(all_pids), "sites": sites,
    }
    (out_dir / "partition.json").write_text(json.dumps(meta, indent=2))
    return meta


train_rows = pd.read_csv(SOURCE_DATASET / "train.csv")
train_patients = train_rows.drop_duplicates("pid")[
    ["pid", "label", "cohort"]].reset_index(drop=True)
print(f"{len(train_patients)} training patients available to divide")

built = {}
for name in BUILD:
    if (PARTITIONS_DIR / name / "partition.json").is_file() and not FORCE_REBUILD:
        print(f"\n{name}: already built — set FORCE_REBUILD = True to redo it")
        built[name] = json.loads((PARTITIONS_DIR / name / "partition.json").read_text())
        continue
    built[name] = build_partition(name, PARTITIONS[name], train_rows,
                                  train_patients, SOURCE_IMAGES, PARTITIONS_DIR)

## How heterogeneous did each partition actually turn out?

This is the cell that tells you whether the experiment you think you built is the
one you built. The class-share spread is the maximum minus the minimum share of any
class across hospitals, in percentage points.

The first four partitions come out under one percentage point, which is the honest
statement that they vary quantity and essentially nothing else. The cohort split
comes out around 27 points. That gap is the reason the cohort pair exists at all.

In [ ]:
rows_out = []
for name, meta in built.items():
    shares = []
    for site in meta["sites"]:
        per_class = np.array(site["train"]["per_class_patients"], dtype=float)
        shares.append(100 * per_class / max(per_class.sum(), 1))
    shares = np.array(shares)
    rows_out.append({
        "partition": name,
        "hospitals": meta["n_clients"],
        "mode": meta["mode"],
        "patients": meta["total_patients"],
        "smallest_site": min(s["train"]["patients"] for s in meta["sites"]),
        "largest_site": max(s["train"]["patients"] for s in meta["sites"]),
        "class_spread_pp": round(float((shares.max(0) - shares.min(0)).max()), 2),
    })

summary = pd.DataFrame(rows_out)
print(summary.to_string(index=False))
print()
print("class_spread_pp is the widest gap, across hospitals, in any one class's share.")
print("Under one point means quantity skew only. The cohort split is the real test.")

## The picture

Class composition per hospital, one panel per partition. The stratified partitions
produce three visually identical bars of different heights, which is exactly the
point being made about them.

In [ ]:
n = len(built)
fig, axes = plt.subplots(1, n, figsize=(3.4 * n, 3.6), sharey=True)
axes = np.atleast_1d(axes)
colours = ["#4c78a8", "#e45756", "#54a24b"]

for ax, (name, meta) in zip(axes, built.items()):
    labels = [s["site"].replace("hospital_", "H") for s in meta["sites"]]
    per_class = np.array([s["train"]["per_class_patients"] for s in meta["sites"]],
                         dtype=float)
    shares = 100 * per_class / np.clip(per_class.sum(1, keepdims=True), 1, None)
    bottom = np.zeros(len(labels))
    for c in range(NUM_CLASSES):
        ax.bar(labels, shares[:, c], bottom=bottom, color=colours[c],
               label=CLASS_NAMES[c] if ax is axes[0] else None)
        bottom += shares[:, c]
    ax.set_title(name.replace("_", " "), fontsize=8)
    ax.set_ylim(0, 100); ax.grid(False)

axes[0].set_ylabel("% of the hospital's training patients")
fig.legend(loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout(); plt.show()

## What this notebook produced

| path | what it is |
|---|---|
| `deployment/data/global/test.csv` | the held-out global test set, 268 patients. Every reported number comes from here |
| `deployment/data/global/val.csv` | the global validation split |
| `deployment/data/global/images/` | hardlinks to the PNGs for those patients |
| `deployment/data/global/manifest.json` | source, build time, per-split counts, trivial baseline |
| `deployment/data/partitions/<name>/hospital_N/train.csv` | that hospital's training rows |
| `deployment/data/partitions/<name>/hospital_N/val.csv` | that hospital's local validation rows |
| `deployment/data/partitions/<name>/hospital_N/images/` | hardlinks to only that hospital's patients |
| `deployment/data/partitions/<name>/hospital_N/manifest.json` | per-split counts and the global class weights |
| `deployment/data/partitions/<name>/partition.json` | the whole split: mode, ratio, seed, source, every site |

**What was verified while building, and raised on failure**

No patient in two hospitals. Every available patient placed. Every patient has an
image folder. Those three are checks rather than assumptions because each of them
has a failure mode that produces a plausible-looking wrong number instead of an
error.

**Where this goes next**

Notebook 07 starts the federation and runs the twelve federated experiments against
these folders.

**One thing worth restating.** The federated arm trains on 1,221 to 1,223 patients
depending on the partition, against 1,527 for the centralised baseline. That is a
fifth less data, and it is a consequence of each hospital holding out its own local
validation split. It is not a flaw in the comparison, but it is part of what any
gap between the two arms is measuring, and the thesis says so.